In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits

from study.data import cohort, features
from study.io import WORK, load, save, write_table
from study.metrics import probability_metrics
from study.models import fit_model, predict
from study.statistics import joint_draws, loss_matrix

In [ ]:
source_fits = load(WORK / 'models/source.pkl')
shared = load(WORK / 'feature_sets.pkl')['shared']
source_x, target_x = features('metavision'), features('eicu')
source_metadata = cohort('metavision').sort_index()
target_metadata = cohort('eicu').query("partition == 'test'").sort_index()

In [ ]:
# Resample source patients and refit each tier
test_ids = target_metadata.index
losses, rows = [], []
for fit, seed in enumerate(range(30)):
    sample = np.random.default_rng(seed).choice(source_metadata.index, size = len(source_metadata), replace = True)
    allocation_losses = []

    for repeat in source_fits['repeats']:
        fit_ids = source_fits['fit_ids']['metavision', repeat]
        training = sample[np.isin(sample, fit_ids)]
        tier_losses = []

        # Refit all tiers on the same resampled source patients
        for tier in ('R1', 'R2', 'R3'):
            with threadpool_limits(limits = 1):
                model = fit_model(
                    source_x.loc[training, shared[tier]],
                    source_metadata.loc[training, 'in_hospital_mortality'],
                    'xgboost',
                    source_fits['parameters'],
                )
                risk = predict(model, target_x.loc[test_ids])

            tier_losses.append(loss_matrix(target_metadata.in_hospital_mortality, risk)[:, 0])
            rows.append(
                dict(
                    fit = fit,
                    seed = seed,
                    repeat = repeat,
                    tier = tier,
                    **probability_metrics(target_metadata.in_hospital_mortality, risk),
                )
            )

        allocation_losses.append(np.stack(tier_losses, axis = 1))

    losses.append(np.mean(allocation_losses, axis = 0))
    
    print('Source bootstrap', fit + 1, 'of 30;', len(source_fits['repeats']), 'paired allocations')

losses = np.stack(losses)
differences = np.diff(losses, axis = 2)
save(WORK / 'predictions/source_bootstraps.pkl', {'ids': test_ids, 'loss_differences': differences})

metrics = pd.DataFrame(rows)
write_table(WORK / 'results/source_refits.parquet', metrics)

In [ ]:
# Summarise source bootstrap distributions
rows = []
for richer, simpler in [('R2', 'R1'), ('R3', 'R2')]:
    # Pair tier performance within each source resample
    averaged = metrics.groupby(['fit', 'tier'])[['log_loss', 'AUROC']].mean().reset_index()
    pivot = averaged.pivot(index = 'fit', columns = 'tier', values = 'log_loss')
    auc = averaged.pivot(index = 'fit', columns = 'tier', values = 'AUROC')
    values = pivot[richer] - pivot[simpler]

    # Summarise the spread and direction of the resampled increments
    rows.append(
        dict(
            contrast = richer + '_minus_' + simpler,
            fits = len(values),
            median = values.median(),
            q10 = values.quantile(0.1),
            q90 = values.quantile(0.9),
            minimum = values.min(),
            maximum = values.max(),
            proportion_favouring_richer = values.lt(0).mean(),
            AUROC_median = (auc[richer] - auc[simpler]).median(),
        )
    )

write_table(WORK / 'results/source_refit_summary.parquet', pd.DataFrame(rows))

In [ ]:
# Combine source resamples with hospital resampling
rows = []
for fits in (15, 30):
    draws = joint_draws(differences[:fits], target_metadata.hospital_id, replicates = 10000, seed = 77)

    # Summarise joint draws with patient and hospital weighting
    for weighting in ('patient', 'equal_hospital'):
        for contrast, name in enumerate(('R2_minus_R1', 'R3_minus_R2')):
            values = draws[weighting][:, contrast]
            rows.append(
                dict(
                    source_fits = fits,
                    contrast = name,
                    weighting = weighting,
                    mean = values.mean(),
                    median = np.median(values),
                    lower = np.quantile(values, 0.025),
                    upper = np.quantile(values, 0.975),
                    fraction_below_zero = (values < 0).mean(),
                )
            )

write_table(WORK / 'results/joint_uncertainty.parquet', pd.DataFrame(rows))

display(pd.DataFrame(rows).round(5))